# C-STDP Synthetic Data Demo

This notebook demonstrates the Causal-STDP algorithm on a synthetic Gene Regulatory Network (GRN).

## Steps
1. Generate Synthetic GRN
2. Simulate Gene Expression with Delays
3. Encode Spikes
4. Run C-STDP Learning Rule
5. Evaluate vs Ground Truth

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
sys.path.append("../")

from src.cstdp.utils.simulate_grn import generate_synthetic_grn, simulate_expression
from src.cstdp.utils.spike_encoding import calculate_adaptive_thresholds, plot_raster
from src.cstdp.core import CausalSTDP
from src.cstdp.utils.evaluate import calculate_metrics

## 1. Generate Data

In [ ]:
np.random.seed(42)
n_genes = 15
n_timepoints = 1000

true_adj, delays = generate_synthetic_grn(n_genes, connection_prob=0.15)
expression = simulate_expression(n_genes, n_timepoints, true_adj, delays, 
                                 noise_level=0.1, burst_prob=0.02, decay=0.2)

plt.figure(figsize=(12, 4))
plt.plot(expression.T[:, :5], alpha=0.7)
plt.title("Gene Expression (First 5 Genes)")
plt.xlabel("Time")
plt.ylabel("Expression")
plt.show()

## 2. Spike Encoding

In [ ]:
thresholds = calculate_adaptive_thresholds(expression, sigma=1.5)
cstdp = CausalSTDP(w_max=1.0, A_pos=0.05, A_neg=0.06, tau_pos=10, tau_neg=10)
time_points = np.arange(n_timepoints)
spike_trains = cstdp.compute_spike_times(expression, time_points, thresholds)

plot_raster(spike_trains, (0, 200))
plt.show()

## 3. Run C-STDP

In [ ]:
inferred_weights = cstdp.run_cstdp(spike_trains, n_genes)

print(f"Max Weight: {np.max(inferred_weights):.4f}")

## 4. Evaluation

In [ ]:
max_w = np.max(inferred_weights)
if max_w > 0:
    inferred_norm = inferred_weights / max_w
else:
    inferred_norm = inferred_weights
    
metrics = calculate_metrics(true_adj, inferred_norm, threshold=0.3)
print(metrics)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(true_adj, ax=axes[0], cmap="viridis", vmin=0, vmax=1)
axes[0].set_title("Ground Truth")
sns.heatmap(inferred_norm, ax=axes[1], cmap="viridis", vmin=0, vmax=1)
axes[1].set_title("Inferred")
plt.show()